# Your First Quantum Circuit

Build, run, and measure a quantum circuit on the local simulator.

**Objectives:**
- Create a quantum circuit using the Braket SDK
- Run it on the local simulator and collect measurement results
- Interpret measurement counts as probability distributions
- Visualize results using the shared library

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->


In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import matplotlib.pyplot as plt

# Use local simulator by default (free, instant)
device = LocalSimulator()

## 1. A Minimal Quantum Circuit

Every quantum circuit starts with qubits initialized to |0>. We apply gates to transform them,
then measure. The simplest interesting circuit applies a Hadamard gate to put a qubit into superposition.

In [ ]:
# Create a 1-qubit circuit: apply Hadamard to qubit 0
circuit = Circuit().h(0)

# Print the circuit diagram
print(circuit)
print(f"\nQubit count: {circuit.qubit_count}")
print(f"Circuit depth: {circuit.depth}")

In [ ]:
# Run the circuit with 1000 shots (repetitions)
result = device.run(circuit, shots=1000).result()

# measurement_counts is a Counter of bitstring -> count
counts = result.measurement_counts
print(f"Measurement results: {dict(counts)}")
print(f"\nTotal shots: {sum(counts.values())}")
print(f"Probability of |0>: {counts.get('0', 0) / 1000:.3f}")
print(f"Probability of |1>: {counts.get('1', 0) / 1000:.3f}")

The Hadamard gate creates an equal superposition: we expect roughly 50% |0> and 50% |1>.
The exact split varies each run due to quantum randomness (simulated here).

## 2. A Two-Qubit Entangled Circuit (Bell Pair)

The Bell pair is the simplest entangled state. After creating it, measuring one qubit
determines the other -- they always agree.

In [ ]:
# Build a Bell pair: H on qubit 0, then CNOT from 0 to 1
bell_circuit = Circuit().h(0).cnot(0, 1)
print(bell_circuit)

# Run and examine results
result = device.run(bell_circuit, shots=1000).result()
counts = result.measurement_counts
print(f"\nResults: {dict(counts)}")
print("\nNotice: only |00> and |11> appear -- the qubits are correlated!")

## 3. Using the Shared Library

The `lib/` package provides reusable circuit patterns and utilities.
Let's use them to build and visualize circuits.

In [ ]:
from lib.circuits import bell_pair, ghz_state
from lib.utils.results import parse_counts, top_results
from lib.utils.visualization import plot_histogram

# Use the library's bell_pair function
circuit = bell_pair()
result = device.run(circuit, shots=2000).result()
counts = parse_counts(result)

print(f"Top results: {top_results(counts, n=5)}")

# Visualize as a histogram
fig = plot_histogram(counts, title="Bell Pair Measurement Results")
plt.show()

In [ ]:
# GHZ state: n-qubit generalization of the Bell pair
ghz = ghz_state(n_qubits=4)
print(ghz)

result = device.run(ghz, shots=2000).result()
counts = parse_counts(result)

fig = plot_histogram(counts, title="4-Qubit GHZ State")
plt.show()
print("Only |0000> and |1111> should appear -- all qubits agree.")

## 4. Exercises

Two exercises to solidify the pattern. Each has tiered hints -- expand only
what you need -- and a check cell that tells you when you have it.


### Exercise 1 — A five-qubit GHZ state

Scale the GHZ pattern up to 5 qubits with the shared library, run it on the
local simulator, and confirm every qubit agrees.

Define `ghz5_counts`: the measurement counts from running your 5-qubit GHZ
circuit with at least 1000 shots.

<details><summary>Hint 1 — nudge</summary>

Section 3 did exactly this for 4 qubits. What actually changes — the circuit
construction, or just one argument?

</details>
<details><summary>Hint 2 — approach</summary>

Build the circuit with `ghz_state(n_qubits=...)`, run it with
`device.run(..., shots=...)`, and turn the result into counts with
`parse_counts(...)` — the same three steps as the 4-qubit GHZ cell above.

</details>


In [ ]:
# Exercise 1: Create a 5-qubit GHZ state and verify all qubits agree.
# Define: ghz5_counts -- measurement counts from at least 1000 shots.

# TODO: your code here


In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert sum(ghz5_counts.values()) >= 1000, "run at least 1000 shots"
    assert all(len(k) == 5 for k in ghz5_counts), "the GHZ state should span 5 qubits"
    assert set(ghz5_counts) <= {"00000", "11111"}, (
        "a GHZ state should only ever measure all-0s or all-1s"
    )
    assert len(ghz5_counts) == 2, "with 1000+ shots you should see BOTH outcomes"


### Exercise 2 — Shots vs accuracy

Measurement is sampling: more shots means a better estimate of the true
probabilities. Quantify that with the single-qubit H circuit from Section 1.

Define `shot_series = [10, 50, 100, 500, 5000]` and `p0_series` — the measured
probability of `|0>` at each shot count, in the same order. Plot them if you
like the picture.

<details><summary>Hint 1 — nudge</summary>

The standard error of a probability estimate shrinks like $1/\sqrt{\text{shots}}$.
What does that predict for the spread of your estimates at 10 shots vs 5000?

</details>
<details><summary>Hint 2 — approach</summary>

Loop over the shot counts. For each: build the 1-qubit `H` circuit, run it
with that many shots, get counts with `parse_counts(...)`, and append
`counts.get("0", 0) / shots` to `p0_series`.

</details>


In [ ]:
# Exercise 2: How does shot count affect measurement accuracy?
# Define: shot_series = [10, 50, 100, 500, 5000], and p0_series -- the
# measured probability of |0> for the 1-qubit H circuit at each shot count.

# TODO: your code here


In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert list(shot_series) == [10, 50, 100, 500, 5000], (
        "use the five shot counts from the prompt"
    )
    assert len(p0_series) == len(shot_series), "record one probability per shot count"
    assert all(0.0 <= p <= 1.0 for p in p0_series), "probabilities live in [0, 1]"
    assert abs(p0_series[-1] - 0.5) < 0.05, (
        "5000 shots should estimate P(|0>) within a few percent of 0.5"
    )


## Summary

- Quantum circuits are built by chaining gate calls: `Circuit().h(0).cnot(0, 1)`
- The `LocalSimulator()` runs circuits instantly at no cost
- Measurement results are probabilistic -- more shots = better statistics
- Entangled qubits (Bell pair, GHZ) show perfect correlations
- The `lib/` package provides reusable patterns: `bell_pair()`, `ghz_state()`, `plot_histogram()`

**Next:** [02-single-qubit-gates.ipynb](02-single-qubit-gates.ipynb) -- explore every single-qubit gate and visualize state transformations.